# KIRA — 3. Dialogue (GPU)

Adapte les poids préentraînés de KIRA au dialogue, puis rejoue les **30 tours de sondes** de conversation
(dont « bsr ») et prépare un fichier allégé à télécharger pour le PC.

**Réglages Kaggle** : *Accelerator* = **GPU T4 x2** ; *Internet* = **On** (sources de dialogue épinglées) ;
*Environment* = **Pin to original environment**.
**Add Input** : la sortie de **kira-1-donnees**, et celles de **kira-2A** et **kira-2B** (la plus avancée est choisie).
Puis **Save Version → Save & Run All (Commit)**.

Les réponses des sondes sont à **relire** : une bonne perte ne prouve pas un bon dialogue.

In [ ]:
# Paramètres
BASE_RUN = "kira-100m-a"          # run de préentraînement à adapter
RUN_NAME = "kira-100m-a-dialogue"
STEPS = 3000
BATCH_SIZE = 64                   # conversations par mise à jour
MICRO_BATCH = 16
LR = 0.0003
WARMUP = 100                      # étapes de montée du taux d'apprentissage (inférieur à STEPS)
REPLAY_RATIO = 0.2                # part des étapes qui répètent le préentraînement
MAX_LANGUAGE_REGRESSION = 0.1     # hausse tolérée de la perte de langue pour retenir des poids
TIME_LIMIT_HOURS = 10.0
KIRA_REF = "main"

In [ ]:
# Retrouve les entrées ajoutées au notebook (« Add Input »), en lecture seule sous /kaggle/input.
import json, os, shutil, subprocess, sys
from pathlib import Path

def find_data():
    found = []
    for manifest in Path("/kaggle/input").rglob("manifest.json"):
        try:
            if json.loads(manifest.read_text(encoding="utf-8")).get("format") == "kira-token-shards-1":
                found.append(manifest.parent)
        except (OSError, ValueError):
            pass
    if len(found) != 1:
        raise SystemExit("Il faut exactement une entrée de données KIRA (sortie du notebook kira-1-donnees). "
                         f"Trouvé : {[str(f) for f in found]}")
    return found[0]

def find_run(name):
    """Session la plus avancée d'un run parmi les entrées : (dossier, étape), ou (None, None)."""
    best, best_step = None, None
    for state in Path("/kaggle/input").rglob(f"runs/{name}/training.pt"):
        manifest = state.parent / "manifest.json"
        step = json.loads(manifest.read_text(encoding="utf-8")).get("steps", 0) if manifest.exists() else 0
        if best_step is None or step > best_step:
            best, best_step = state.parent, step
    return best, best_step

In [ ]:
# Récupère le code public de KIRA au commit choisi et installe le tokenizer épinglé.
def run(*command, **options):
    print("$", " ".join(map(str, command)), flush=True)
    subprocess.run([str(c) for c in command], check=True, **options)

CODE = Path("/tmp/KIRA")
if not (CODE / ".git").exists():
    run("git", "clone", "--quiet", "https://github.com/BlackGhost99/KIRA", CODE)
run("git", "-C", CODE, "fetch", "--quiet", "origin")
run("git", "-C", CODE, "checkout", "--quiet", "--detach", KIRA_REF if len(KIRA_REF) == 40 else "origin/" + KIRA_REF)
COMMIT = subprocess.run(["git", "-C", str(CODE), "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
os.environ.update(KIRA_CODE_COMMIT=COMMIT, PYTHONPATH=str(CODE), PYTHONUNBUFFERED="1")
run(sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", "tokenizers==0.23.3")
import torch
image = Path("/etc/git_commit")
print("Code KIRA :", COMMIT, "| PyTorch", torch.__version__, "| GPU :", torch.cuda.device_count(),
      "| image Kaggle :", image.read_text().strip() if image.exists() else "?")

In [ ]:
# Lance une commande et garde aussi sa sortie dans un fichier journal.
def stream(command, log_path):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print("$", " ".join(map(str, command)), flush=True)
    with open(log_path, "a", encoding="utf-8") as log:
        process = subprocess.Popen([str(c) for c in command], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
        code = process.wait()
    if code:
        raise SystemExit(f"Échec (code {code}) : voir {log_path}. Les sauvegardes déjà écrites restent dans /kaggle/working.")

In [ ]:
DATA = find_data()
BASE, BASE_STEP = find_run(BASE_RUN)
if BASE is None or not (BASE / "weights.pt").exists():
    raise SystemExit(f"Poids {BASE_RUN}/weights.pt introuvables : ajouter les sorties de kira-2A et kira-2B en entrée.")
print("Poids de départ :", BASE, "étape", BASE_STEP)
CONTEXT = json.loads((BASE / "manifest.json").read_text(encoding="utf-8"))["config"]["context"]
CONVERSATIONS = Path("/kaggle/working/conversations/conversations.jsonl")
if not CONVERSATIONS.exists():
    stream([sys.executable, "-m", "research.kira_model.sft_data", "prepare", "--tokenizer", DATA / "tokenizer.json",
            "--context", CONTEXT, "--raw-dir", Path("/kaggle/working/brut-dialogue"), "--output", CONVERSATIONS],
           Path("/kaggle/working/conversations/journal.txt"))
    shutil.rmtree("/kaggle/working/brut-dialogue", ignore_errors=True)
print(json.loads(CONVERSATIONS.with_suffix(".manifest.json").read_text(encoding="utf-8"))["examples"])

In [ ]:
GPUS = torch.cuda.device_count()
if not GPUS:
    raise SystemExit("Aucun GPU visible : choisir un accélérateur GPU dans les réglages du notebook.")
if BATCH_SIZE % (MICRO_BATCH * GPUS):
    GPUS = 1  # le lot doit se répartir exactement entre les GPU
OUTPUT = Path("/kaggle/working/runs") / RUN_NAME
arguments = ["sft", "--base", BASE / "weights.pt", "--conversations", CONVERSATIONS, "--replay-data", DATA,
             "--output", OUTPUT, "--steps", STEPS, "--batch-size", BATCH_SIZE, "--micro-batch", MICRO_BATCH,
             "--lr", LR, "--warmup", WARMUP, "--replay-ratio", REPLAY_RATIO, "--max-language-regression", MAX_LANGUAGE_REGRESSION,
             "--time-limit", int(TIME_LIMIT_HOURS * 3600)]
launcher = ([sys.executable, "-m", "torch.distributed.run", "--standalone", "--nproc_per_node", GPUS]
            if GPUS > 1 else [sys.executable])
# Journal à part : le dossier du run ne doit contenir que ce que l'entraîneur écrit.
stream([*launcher, "-m", "research.kira_model.gpu_train", *arguments], Path("/kaggle/working/journaux") / f"{RUN_NAME}.txt")

In [ ]:
# Sondes de conversation (décodage glouton, comme au laboratoire) et fichier allégé à télécharger.
REPORTS = Path("/kaggle/working/rapports")
report = REPORTS / f"{RUN_NAME}-sondes.json"
if not report.exists():
    stream([sys.executable, "-m", "research.kira_model.lm_chat", "probe", "--weights", OUTPUT / "weights.pt",
            "--output", report, "--max-new-tokens", 64], REPORTS / "journal.txt")
light = Path("/kaggle/working") / f"{RUN_NAME}-inference.pt"
if not light.exists():
    stream([sys.executable, "-m", "research.kira_model.lm_chat", "export", "--weights", OUTPUT / "weights.pt",
            "--output", light], REPORTS / "journal.txt")
for case in json.loads(report.read_text(encoding="utf-8"))["cases"]:
    for turn in case["turns"]:
        print(f"Toi > {turn['user']}\nKIRA > {turn['response']}\n")